In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Circle
import pandas as pd
from scipy.optimize import curve_fit


# -------------------------
# Loading images
# -------------------------
img_paths = {
    "2h":  "2_hours.PNG",
    "24h": "24_hours.PNG",
    "70h": "70_hours.PNG",
}
images = {k: np.array(Image.open(p).convert("RGB")) for k, p in img_paths.items()}

# -------------------------
# ROI settings
# x = column, y = row
# -------------------------
radius = 32
bg_offset = (0, -95)

# Choosing spot centres
spot_centers = {
    "2h":  (190, 180),
    "24h": (180, 195),
    "70h": (171, 130),
}

# -------------------------
# Circle pixel extraction
# -------------------------
def extract_circle_pixels(img_rgb, cx, cy, r):
    H, W, _ = img_rgb.shape
    Y, X = np.ogrid[:H, :W]
    mask = (X - cx)**2 + (Y - cy)**2 <= r**2
    return img_rgb[mask]  # (N,3)

# -------------------------
# 1) Showing images with circles
# -------------------------
for lab, img in images.items():
    sx, sy = spot_centers[lab]
    bx = sx + bg_offset[0]
    by = sy + bg_offset[1]

    fig, ax = plt.subplots(figsize=(5,5))
    ax.imshow(img)
    ax.add_patch(Circle((sx, sy), radius, fill=False, edgecolor="red", linewidth=2))
    ax.add_patch(Circle((bx, by), radius, fill=False, edgecolor="blue", linewidth=2))
    ax.set_title(f"{lab}: red=spot ROI, blue=background ROI")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

# -------------------------
# 2) Collecting exposed + background pixels
# -------------------------
exposed_pixels = {}
for lab, img in images.items():
    sx, sy = spot_centers[lab]
    exposed_pixels[lab] = extract_circle_pixels(img, sx, sy, radius).astype(np.uint8)


sx2, sy2 = spot_centers["2h"]
bx2 = sx2 + bg_offset[0]
by2 = sy2 + bg_offset[1]
bg_pixels = extract_circle_pixels(images["2h"], bx2, by2, radius).astype(np.uint8)


# -------------------------
# 3) Histogram plotting + RAW DATA extraction
# -------------------------
bin_edges = np.arange(0, 256, 2)

order = ["2h", "24h", "70h"]  # plot order

hist_data = {}  # store counts + bin edges for each channel & time
def do_channel(channel_index, channel_name):
    channel_colors = {
        "R": (0.6350, 0.0780, 0.1840),
        "G": (0.4660, 0.6740, 0.1880),
        "B": (0.25, 0.35, 0.85),
    }
    col = channel_colors[channel_name]

    fig, axes = plt.subplots(4, 1, figsize=(7, 10), sharex=True)
    fig.suptitle(f"{channel_name} channel histograms", y=0.995)

    for i, lab in enumerate(order):
        vals = exposed_pixels[lab][:, channel_index]
        counts, edges = np.histogram(vals, bins=bin_edges)

        hist_data[f"{lab}_{channel_name}_Values"] = counts
        hist_data[f"{lab}_{channel_name}_BinEdges"] = edges

        axes[i].hist(vals, bins=bin_edges, color=col, edgecolor="k")
        axes[i].legend([f"{channel_name} {lab}"], loc="upper right")
        axes[i].set_ylabel("Count")

    vals0 = bg_pixels[:, channel_index]
    counts0, edges0 = np.histogram(vals0, bins=bin_edges)
    hist_data[f"bg_{channel_name}_Values"] = counts0
    hist_data[f"bg_{channel_name}_BinEdges"] = edges0

    axes[3].hist(vals0, bins=bin_edges, color=col, edgecolor="k")
    axes[3].legend([f"{channel_name} background"], loc="upper right")
    axes[3].set_xlabel("Pixel value")
    axes[3].set_ylabel("Count")

    plt.tight_layout()
    plt.show()

do_channel(0, "R")
do_channel(1, "G")
do_channel(2, "B")

# -------------------------
# 4) Exporting the raw histogram numbers
#    bin centres + counts for each histogram
# -------------------------

edges_ref = hist_data["2h_R_BinEdges"]
bin_centres = 0.5 * (edges_ref[:-1] + edges_ref[1:])

out = pd.DataFrame({"bin_center": bin_centres})

for key, arr in hist_data.items():
    if key.endswith("_Values"):
        out[key] = arr

out.to_csv("histogram_values.csv", index=False)
print("Saved: histogram_values.csv")

# ============================================================
# 5) Mean RGB per exposure + scatter plots
# ============================================================

# ROI pixels table
rows = []
for lab in order:
    spot = exposed_pixels[lab].astype(np.float32)
    mean_rgb = spot.mean(axis=0)
    rows.append({
        "time_h": int(lab.replace("h","")),
        "spot_R": mean_rgb[0],
        "spot_G": mean_rgb[1],
        "spot_B": mean_rgb[2],
    })

df = pd.DataFrame(rows).sort_values("time_h").reset_index(drop=True)
print(df)

# ============================================================
# 6) DELTA (Exposure - Background)
# ============================================================

# Compute mean background RGB (from your bg_pixels)
bg_mean = bg_pixels.astype(np.float32).mean(axis=0)

# Add delta columns to df
df["delta_R"] = df["spot_R"] - bg_mean[0]
df["delta_G"] = df["spot_G"] - bg_mean[1]
df["delta_B"] = df["spot_B"] - bg_mean[2]


print("\nDelta values (spot - background):")
print(df[["time_h", "delta_R", "delta_G", "delta_B"]])

FileNotFoundError: [Errno 2] No such file or directory: '2_hours.PNG'